# 🚀 Salesforce CRM Notes Formatter — Enterprise QLoRA Fine-Tuning

### 📌 Project Overview
This notebook implements an end-to-end pipeline for fine-tuning **Google Gemma-2B-IT** using **QLoRA (Quantized Low-Rank Adaptation)** to automatically transform messy, unstructured sales meeting notes into strictly structured, enterprise-ready Salesforce CRM sections:
1. `Summary:` (Key meeting highlights and stakeholders)
2. `Key Pain Points:` (Customer obstacles, legacy tool limitations, budget constraints)
3. `Action Items:` (Committed deliverables and owners)
4. `Next Steps:` (Follow-up timelines and milestone dates)
5. `Date/Time of Interaction:` (Standardized audit timestamp)

---

### ⚙️ Technical Architecture & Key Innovations
- **Base Model**: `google/gemma-2b-it` (2.5 Billion parameter instruction-tuned causal LLM).
- **Quantization (QLoRA)**: 4-bit NormalFloat (`nf4`) with double quantization and FP16 compute dtype via `bitsandbytes`.
- **Parameter Efficiency (LoRA)**: Freezes 99.2% of base model weights; injects low-rank trainable adapter matrices ($r=16, \alpha=32$) into all 7 linear projection layers (`q, k, v, o, gate, up, down`).
- **VRAM Optimization for 4GB GPUs**: Paged AdamW 8-bit optimizer (`paged_adamw_8bit`) + Gradient Checkpointing + Gradient Accumulation (Effective batch size = 4).
- **Enterprise Security**: Secure token management preventing credential leakage.

In [1]:
# Step 0: Install necessary enterprise LLM fine-tuning libraries
# - transformers: Hugging Face model architectures and pipelines
# - peft: Parameter-Efficient Fine-Tuning (LoRA/QLoRA)
# - trl: Transformer Reinforcement Learning & Supervised Fine-Tuning (SFTTrainer)
# - datasets: Memory-mapped dataset loading and preprocessing
# - bitsandbytes: 4-bit and 8-bit quantization CUDA kernels
# - accelerate: Hardware acceleration and memory-efficient dispatch
!pip install -q -U transformers peft trl datasets bitsandbytes accelerate


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Security & Hardware Verification
### 🔒 Security Best Practice: Token Management
Hardcoding API keys inside source code or Jupyter notebooks is a major security vulnerability. We securely load `HF_TOKEN` from the operating system environment variable, with a fallback prompt if not set.

In [2]:
import os
import torch

# 1. Secure Token Retrieval (Automatic fallback to workspace .env file)
if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"]:
    env_paths = [".env", "../.env", "Notebook/.env"]
    for path in env_paths:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line.startswith("HF_TOKEN="):
                        os.environ["HF_TOKEN"] = line.split("=", 1)[1].strip()
                        break
            if "HF_TOKEN" in os.environ:
                break

if os.getenv("HF_TOKEN"):
    print("✓ Hugging Face authentication token loaded successfully.")
else:
    print("Notice: HF_TOKEN not found in environment. Ensure HF_TOKEN is defined if accessing gated models.")

# 2. Verify GPU Acceleration and CUDA Compute Capability
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Active GPU:      {device_name} ({total_vram:.2f} GB VRAM)")
else:
    print("WARNING: CUDA is not available. GPU acceleration is required for 4-bit QLoRA training.")


✓ Hugging Face authentication token loaded successfully.
PyTorch Version: 2.5.1+cu121
CUDA Available:  True
Active GPU:      NVIDIA GeForce GTX 1650 Ti (4.00 GB VRAM)


## 2. Dataset Preparation & ChatML Formatting

### 💬 ChatML Structure for Gemma
Instruction-tuned models like Gemma utilize standardized chat templates. Because Gemma's native Jinja template requires alternating `user` and `model` roles, we prepend our **System Prompt** directly into the first `user` turn.

This guides the LLM to strictly extract and format into our standard CRM schema.

In [3]:
import os
from datasets import load_dataset

# 1. Robust path resolution for workspace and subdirectories
data_path = "train.jsonl" if os.path.exists("train.jsonl") else "Notebook/train.jsonl"
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Training dataset not found at {data_path}.")

# 2. Load JSONL dataset
dataset = load_dataset("json", data_files=data_path, split="train")
print(f"Successfully loaded {len(dataset)} training examples from '{data_path}'.")

# 3. Define the Enterprise Salesforce System Prompt
SYSTEM_PROMPT = """You are an expert Salesforce CRM Assistant. Your task is to take raw, messy meeting notes and format them strictly according to the company's best practices.
You must extract and organize the information into the following sections exactly:
Summary:
Key Pain Points:
Action Items:
Next Steps:
Date/Time of Interaction:
"""

# 4. Format dataset into ChatML conversational turns
def format_to_chatml(example):
    return {
        "messages": [
            {"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{example['input']}"},
            {"role": "assistant", "content": example["output"]}
        ]
    }

chat_dataset = dataset.map(format_to_chatml, remove_columns=["input", "output"])
print("\nSample Formatted Training Record (ChatML):")
print(chat_dataset[0]["messages"])

d:\Personal Project_certificate work Space(Projects )\salesforce project\salesforce-integration\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully loaded 40 training examples from 'train.jsonl'.


Map: 100%|██████████| 40/40 [00:00<00:00, 961.57 examples/s]


Sample Formatted Training Record (ChatML):
[{'role': 'user', 'content': "You are an expert Salesforce CRM Assistant. Your task is to take raw, messy meeting notes and format them strictly according to the company's best practices.\nYou must extract and organize the information into the following sections exactly:\nSummary:\nKey Pain Points:\nAction Items:\nNext Steps:\nDate/Time of Interaction:\n\n\nAcme Corp – CTO Sam Green: \n○ Explored current infrastructure; using legacy CRM. \nInterested in reducing manual data entry by 40%. \nIdentified budget stage and decision timeline (Q4)."}, {'role': 'assistant', 'content': 'Summary\n• Met with Sam Green, CTO at Acme Corp\n• Discussed current infrastructure and legacy CRM\n\nKey Pain Points\n• Interested in reducing manual data entry by 40%\n• Current system requires excessive manual labor\n\nAction Items\n• Work on automating CRM processes to reduce manual input\n• Align with budget stage and decision timeline (Q4)\n\nNext Steps\n• Schedul

## 3. Model & Tokenizer Initialization (QLoRA 4-bit)

### 🧠 How QLoRA Fits into 4GB VRAM
A standard FP16 2.5B parameter model requires ~5GB VRAM just to store weights in memory, causing Out-Of-Memory (OOM) during training on 4GB GPUs. 

QLoRA resolves this through **three key mechanisms**:
1. **NF4 (4-bit NormalFloat)**: Quantizes weights into an information-theoretically optimal distribution for normally distributed neural network weights (~1.4GB VRAM footprint).
2. **Double Quantization**: Quantizes the quantization constants themselves, saving ~0.37 bits per parameter.
3. **FP16 Compute Dtype**: Dequantizes 4-bit weights into 16-bit floats dynamically during the forward pass for maximum precision.

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-2b-it"

# 1. Configure BitsAndBytes 4-Bit NormalFloat Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. Load Tokenizer & configure padding token
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Crucial for causal LM batch training

# 3. Load Quantized Base Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Disable KV caching during training (required for gradient checkpointing)
model.config.use_cache = False
print("Base model successfully loaded with 4-bit quantization!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 164/164 [00:15<00:00, 10.47it/s]


Base model successfully loaded with 4-bit quantization!


## 4. LoRA Adapter Configuration (PEFT)

### 🎯 Low-Rank Adaptation Mathematics
Instead of updating the full weight matrix $W_0 \in \mathbb{R}^{d \times k}$, LoRA decomposes the weight update into two low-rank matrices:
$$W = W_0 + \Delta W = W_0 + \frac{\alpha}{r} (B \times A)$$
Where:
- $A \in \mathbb{R}^{r \times k}$ and $B \in \mathbb{R}^{d \times r}$ with rank $r \ll \min(d, k)$.
- **Rank ($r = 16$)**: Determines the dimensionality of the adaptation subspace.
- **Alpha ($\alpha = 32$)**: Scaling factor balancing the influence of the adapter.
- **Target Modules**: Injected into all 7 linear projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).

This trains only **~0.78% of the total model parameters** (~19.6 Million out of 2.5 Billion), drastically cutting gradient memory!

In [5]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Prepare quantized model for k-bit training (stabilizes layer norms and enables gradient checkpointing)
model = prepare_model_for_kbit_training(model)

# 2. Define LoRA Hyperparameters
peft_config = LoraConfig(
    r=16,                                    # Rank dimension
    lora_alpha=32,                           # Alpha scaling factor
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,                       # Dropout for regularization
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Attach trainable LoRA adapters to frozen base model
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 19,611,648 || all params: 2,525,784,064 || trainable%: 0.7765


## 5. Supervised Fine-Tuning (SFTTrainer)

### 🛠️ Hardware Optimizations Explained
- **`paged_adamw_8bit`**: Uses CUDA memory paging to automatically offload optimizer states between GPU and CPU RAM during memory spikes.
- **`gradient_checkpointing=True`**: Trades compute for memory by recalculating intermediate activations during the backward pass instead of caching them.
- **`gradient_accumulation_steps=4`**: Accumulates gradients over 4 micro-batches of size 1, achieving an effective batch size of 4 without VRAM overhead.
- **`fp16=False` & `bf16=False`**: Avoids PyTorch `GradScaler` non-finite check crashes on Turing GPUs (GTX 1650 Ti lack native BF16 instructions) while computation remains FP16 inside `bitsandbytes` kernels.

In [ ]:
from trl import SFTTrainer, SFTConfig

# 1. Configure Fine-Tuning Hyperparameters
training_args = SFTConfig(
    output_dir="./salesforce-gemma-lora",
    num_train_epochs=3,            # 3 Full passes over CRM meeting dataset
    per_device_train_batch_size=1, # Minimal batch size to protect 4GB VRAM
    gradient_accumulation_steps=4, # Effective batch size = 1 * 4 = 4
    gradient_checkpointing=True,   # Saves ~60% activation memory
    optim="paged_adamw_8bit",      # Paged 8-bit memory-efficient optimizer
    learning_rate=2e-4,            # Standard LoRA learning rate
    fp16=False,                    # QLoRA 4-bit compute already runs in float16
    bf16=False,                    # GTX 1650 Ti does not support bfloat16
    max_grad_norm=0.3,             # Gradient clipping to prevent exploding gradients
    warmup_steps=10,               # Linear warmup for stability
    logging_steps=5,               # Log training loss every 5 steps
    save_strategy="epoch",         # Checkpoint model per epoch
    dataset_text_field="messages",
    max_length=256
)

# 2. Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=chat_dataset,
    args=training_args,
    processing_class=tokenizer
)

# 3. Execute Training Loop
print("Starting Supervised Fine-Tuning...")
trainer.train()

Truncating train dataset: 100%|██████████| 40/40 [00:00<00:00, 1113.91 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 40/40 [00:00<00:00, 1599.31 examples/s]


Starting Supervised Fine-Tuning...


Step,Training Loss
5,4.382969


## 6. Save Adapter & Interactive Inference Engine
We save the lightweight LoRA adapter (~39.2 MB) along with tokenizer configurations. We then re-enable Key-Value caching (`model.config.use_cache = True`) to accelerate autoregressive generation during inference.

In [ ]:
# 1. Save LoRA Adapter and Tokenizer
adapter_dir = "salesforce_notes_adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"✓ LoRA adapter successfully saved to: ./{adapter_dir}/")

# 2. Switch Model to Evaluation Mode
model.eval()
model.config.use_cache = True # Enable KV cache for fast inference

# 3. CRM Notes Formatting Inference Function
def format_crm_notes(raw_notes: str) -> str:
    """
    Formats raw, unstructured notes into enterprise Salesforce CRM sections.
    """
    prompt = tokenizer.apply_chat_template([
        {"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{raw_notes}"}
    ], tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.15,
            eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<end_of_turn>")],
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Extract only the generated assistant tokens
    input_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
    return response

# 4. Test with a sample raw meeting note
sample_input = """TechGlobal - VP of Sales John Doe:
Was complaining about the lack of pipeline visibility.
They are evaluating 3 different CRM solutions right now.
Wants me to send him the pricing sheet by Friday."""

print("\n--- Formatted Output ---")
print(format_crm_notes(sample_input))

## 7. Multi-Scenario Enterprise Benchmark Suite

### 📊 Automated Schema Compliance Scoring
We systematically test the model against 3 diverse real-world edge cases:
1. **Enterprise Healthcare Migration** (High-stakes migration timeline & latency issues)
2. **Urgent Production Escalation** (Critical bug / lead conversion failure)
3. **Messy Shorthand Note** (Shorthand slang, lowercase syntax, API rate limit issues)

The automated evaluator scores each response for adherence to the 5 mandatory CRM sections.

In [ ]:
# 1. Define Multi-Scenario Enterprise Test Cases
test_cases = [
    {
        "title": "Enterprise Healthcare Migration",
        "raw": "Client: Apex Health. Met with CTO Sarah on Oct 12 at 2 PM. Complained their legacy CRM takes 4 minutes to load patient files. Looking to migrate 500 users to Salesforce Health Cloud by Q1. Need a migration quote and architectural review by next Wednesday."
    },
    {
        "title": "Urgent Production Escalation",
        "raw": "Call with Mike from Global Logistics at 9:30 AM today. Their sales reps cannot log lead conversions since yesterday update. Loss of estimated $50k in deals. Escalated to Tier 3 engineering team immediately. Scheduled an hourly update sync with Mike until resolution."
    },
    {
        "title": "Messy Quick Shorthand Note",
        "raw": "quick sync w/ dev team lead alex (11/05 4pm). salesforce api rate limit exceeded 3 times this week during batch sync. blocker for nightly billing. alex will optimize soql queries + batch size tonight. will check error logs tmrw mrng."
    }
]

# 2. Schema Compliance Evaluator
def evaluate_schema(output_text: str):
    required_sections = ["Summary:", "Key Pain Points:", "Action Items:", "Next Steps:", "Date/Time of Interaction:"]
    results = {}
    for sec in required_sections:
        clean_sec = sec.lower().replace(":", "")
        results[sec] = clean_sec in output_text.lower().replace(":", "")
    compliance_score = (sum(results.values()) / len(required_sections)) * 100
    return compliance_score, results

# 3. Run Benchmark Suite
print("=== STARTING ENTERPRISE TEST BENCHMARK SUITE ===\n")
total_score = 0

for i, test in enumerate(test_cases, 1):
    print("=" * 65)
    print(f"[TEST CASE {i}] {test['title']}")
    print(f"RAW INPUT:\n{test['raw']}\n")
    formatted_result = format_crm_notes(test["raw"])
    print(f"FORMATTED OUTPUT:\n{formatted_result}\n")
    score, details = evaluate_schema(formatted_result)
    total_score += score
    print(f"Schema Compliance: {score:.0f}%")
    for sec, passed in details.items():
        status = "[✓ PASS]" if passed else "[✗ FAIL]"
        print(f"   {status} {sec}")
    print("=" * 65 + "\n")

overall_avg = total_score / len(test_cases)
print(f"=== OVERALL BENCHMARK COMPLIANCE SCORE: {overall_avg:.1f}% ===")